# 01 — Build USL-Suspilne Dataset

End-to-end pipeline: Firebase → annotations → download videos → split → crop signer.

**Output:** `data/usl-suspilne/` with `annotations.csv` and `features/{videoId}/{clip}.mp4`

**Requirements:** `pip install firebase-admin yt-dlp`, `ffmpeg` installed, `serviceAccountKey.json` in repo root.

In [ ]:
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    ROOT = Path("/content/ucu-text-to-sign")
    ROOT.mkdir(parents=True, exist_ok=True)
else:
    ROOT = Path(".").resolve().parent

sys.path.insert(0, str(ROOT / "scripts"))

# All paths in one place
FIREBASE_DIR = ROOT / "data/firebase"
ANNOTATIONS_CSV = ROOT / "data/usl-suspilne/annotations.csv"
SPLITS_CSV = ROOT / "data/cache/splits.csv"
RAW_VIDEOS_DIR = ROOT / "data/cache/raw_videos"
UNCROPPED_DIR = ROOT / "data/cache/uncropped"
FEATURES_DIR = ROOT / "data/usl-suspilne/features"

print(f"Project root: {ROOT}")
print(f"Environment: {'Colab' if IN_COLAB else 'Local'}")

## 1. Download Firebase Export

In [ ]:
from download_firebase import download_and_save

export_path = download_and_save(output_dir=FIREBASE_DIR)
print(f"\nExport saved to: {export_path}")

## 2. Build Annotations

Reads the Firebase export (data/firebase/latest.json), finds all captions marked as `aligned`, and writes:
- `data/usl-suspilne/annotations.csv` — `name|text|annotator` (for training)
- `data/cache/splits.csv` — `name|video|start|end` (for download/split)

Filters out clips shorter than 0.3s and any excluded video IDs.

In [ ]:
from build_annotations_from_firebase import build_annotations

result = build_annotations(
    export_path=FIREBASE_DIR / "latest.json",
    output_csv=ANNOTATIONS_CSV,
    splits_csv=SPLITS_CSV,
)
print(f"\n{result['rows']} annotations from {len(result['videos'])} videos")

## 3. Download Videos

Downloads YouTube videos at best available quality.
Skips already-downloaded videos. Use `--force VIDEO_ID` to re-download.

In [ ]:
from download_videos import download_all

dl_result = download_all(splits_csv=SPLITS_CSV, dst=RAW_VIDEOS_DIR)

if dl_result["failed"]:
    print(f"\nFailed videos will be excluded in the verify step.")

## 4. Split into Clips

Splits videos into sentence-level clips based on timestamps.
Skips already-split clips. Use `--force VIDEO_ID` to re-split.

In [ ]:
from split_videos import split_all

split_result = split_all(splits_csv=SPLITS_CSV, raw_dir=RAW_VIDEOS_DIR, dst=UNCROPPED_DIR)

## 5. Crop to Signer Region

Crops the bottom-right 510x510 region where the sign language interpreter appears.

In [ ]:
from crop_signer import crop_all

crop_result = crop_all(src=UNCROPPED_DIR, dst=FEATURES_DIR)

## 6. Verify Dataset

In [ ]:
import csv
import subprocess
import json

dataset_dir = ROOT / "data/usl-suspilne"

# Count annotations
with open(ANNOTATIONS_CSV) as f:
    annotations = list(csv.DictReader(f, delimiter="|"))
print(f"Annotations: {len(annotations)}")

# Count clips per video
for vid_dir in sorted(FEATURES_DIR.iterdir()):
    if vid_dir.is_dir():
        clips = list(vid_dir.glob("*.mp4"))
        print(f"  {vid_dir.name}: {len(clips)} clips")

total_clips = len(list(FEATURES_DIR.glob("*/*.mp4")))
print(f"\nTotal clips: {total_clips}")

# Check for missing clips and filter annotations
missing = []
valid = []
for row in annotations:
    clip = FEATURES_DIR / row["name"].split("/")[0] / (row["name"].split("/")[1] + ".mp4")
    if clip.exists():
        valid.append(row)
    else:
        missing.append(row["name"])

if missing:
    print(f"\nWARNING: {len(missing)} annotations without clips — removing from annotations.csv")
    missing_vids = {}
    for name in missing:
        vid = name.split("/")[0]
        missing_vids[vid] = missing_vids.get(vid, 0) + 1
    for vid, count in sorted(missing_vids.items()):
        print(f"  {vid}: {count} missing")

    with open(ANNOTATIONS_CSV, "w", newline="", encoding="utf-8") as f:
        f.write("name|text|annotator\n")
        for row in valid:
            f.write(f"{row['name']}|{row['text']}|{row['annotator']}\n")
    print(f"\nFiltered annotations: {len(valid)} (removed {len(missing)})")
else:
    print("\nAll annotations have matching clips.")

In [ ]:
# Total duration
total_dur = 0
per_video = {}
for clip in sorted(FEATURES_DIR.glob("*/*.mp4")):
    result = subprocess.run(
        ["ffprobe", "-v", "quiet", "-print_format", "json", "-show_format", str(clip)],
        capture_output=True, text=True
    )
    dur = float(json.loads(result.stdout)["format"]["duration"])
    total_dur += dur
    vid = clip.parent.name
    per_video[vid] = per_video.get(vid, 0) + dur

for vid in sorted(per_video):
    m, s = divmod(per_video[vid], 60)
    print(f"  {vid}: {int(m)}m {s:.0f}s")

h, remainder = divmod(total_dur, 3600)
m, s = divmod(remainder, 60)
print(f"\nTotal duration: {int(h)}h {int(m)}m {s:.0f}s")

## 7. Archive & Upload to Google Drive

Creates a tar.gz archive of the dataset and uploads it to Google Drive for use in Colab notebooks.

In [ ]:
import subprocess

archive_path = ROOT / "data" / "usl-dataset.tar.gz"
subprocess.run(
    ["tar", "czf", str(archive_path),
     "--exclude", "._*",
     "-C", str(ROOT),
     "data/usl-suspilne/annotations.csv", "data/usl-suspilne/features",
     "scripts",
     "experiments/progressive_transformers"],
    check=True,
)

size_mb = archive_path.stat().st_size / 1024 / 1024
print(f"Archive: {archive_path} ({size_mb:.0f} MB)")
print("Contents: data/usl-suspilne/ + scripts/ + experiments/progressive_transformers/")

In [11]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    DRIVE_DIR = Path("/content/drive/MyDrive/ucu-text-to-sign")
    DRIVE_DIR.mkdir(parents=True, exist_ok=True)

    import shutil
    shutil.copy2(archive_path, DRIVE_DIR / archive_path.name)
    print(f"Uploaded to Google Drive: {DRIVE_DIR / archive_path.name}")
else:
    print(f"Not in Colab — upload {archive_path.name} to Google Drive manually,"
          f" or run this notebook in Colab.")

Not in Colab — upload usl-dataset.tar.gz to Google Drive manually, or run this notebook in Colab.
